# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ailya-Shah/INTERNSHIP-TASKS/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup -- rebuild the exact same data, rule baseline, and split as w04

Same March 2026 slice, same honest feature set, same rule score (with the fixed medians + continuous tiebreak), same `GroupShuffleSplit(random_state=42)` -- so the model-vs-baseline comparison below is on identical ground, not a new split that could quietly favor either side.

In [1]:
%pip install -q duckdb huggingface_hub


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, getpass
import duckdb
import numpy as np
import pandas as pd
from scipy.stats import rankdata

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT    = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
MONTH   = "2026-03"

page_agg = con.sql(f"""
    SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
           SUM(gsc_impressions) AS impressions_win,
           SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_position_win
    FROM {FACT}
    WHERE month = '{MONTH}' AND gsc_data_available = TRUE
    GROUP BY content_hash_id
""").df()
page_agg["is_page_one"] = page_agg["avg_position_win"].between(1, 10).astype(int)
page_agg = page_agg[page_agg["impressions_win"] >= 100]

content = con.sql(f"""
    SELECT content_hash_id, word_count, search_volume, competition, backlinks, content_type, main_intent,
           GREATEST(date_diff('day', content_created_date, DATE '{MONTH}-01'), 0) AS content_age_days
    FROM {CONTENT}
    WHERE is_published = TRUE AND is_deleted = FALSE
""").df()

data = page_agg.merge(content, on="content_hash_id", how="left").dropna(
    subset=["word_count", "search_volume", "competition", "backlinks"]
)
print(f"modeling frame: {len(data):,} pages | base rate: {data['is_page_one'].mean():.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

modeling frame: 53,892 pages | base rate: 56.2%


In [3]:
# Rebuild the exact w04 rule baseline (fixed medians + continuous tiebreak) for a fair comparison later
def positive_median(s):
    pos = s[s > 0]
    return pos.median() if len(pos) else s.median()

med_search_volume = positive_median(data["search_volume"])
med_competition   = positive_median(data["competition"])
med_backlinks     = positive_median(data["backlinks"])

data["high_demand"]      = (data["search_volume"] >= med_search_volume).astype(int)
data["low_competition"]  = (data["competition"]   <= med_competition).astype(int)
data["strong_backlinks"] = (data["backlinks"]     >= med_backlinks).astype(int)
data["established"]      = (data["content_age_days"] >= 90).astype(int)
data["rule_score"] = data[["high_demand", "low_competition", "strong_backlinks", "established"]].sum(axis=1)

data["backlinks_rank"]       = rankdata(data["backlinks"])
data["search_volume_rank"]   = rankdata(data["search_volume"])
data["low_competition_rank"] = rankdata(-data["competition"])
data["rule_tiebreak"] = (data["backlinks_rank"] + data["search_volume_rank"] + data["low_competition_rank"]) / 3

print("Rule baseline rebuilt -- same medians, same tiebreak logic as w04.")


Rule baseline rebuilt -- same medians, same tiebreak logic as w04.


## 1. Method choice and why

**Logistic Regression first, then Random Forest -- per the training-honest-models skill's "readable -> stronger" order.**

Logistic Regression is my primary model for interpretation: its coefficients (once features are standardized) are directly readable as "this signal pushes toward/away from page-one, by this much" -- which is exactly what a Ranking Signal Analysis paper needs to report. Random Forest is trained second, purely as a strength check: if it meaningfully outperforms logistic regression, that's evidence the true relationship has real nonlinearity/interaction the linear model can't capture (which w04's signal audit hints at -- length showed a strong non-monotonic-looking drop, not a clean linear trend). Either way, permutation importance on both models gives the ranked "which signals matter" story that this lane's deliverable actually needs -- the model itself is secondary to that ranking.

In [4]:
# Reasoning-only -- the honest feature set, carried forward from w03b's leakage hunt (query-mix and
# clicks_win/avg_position_win stay excluded; they were confirmed leaks).
NUM_FEATURES = ["word_count", "content_age_days", "search_volume", "competition", "backlinks"]
CAT_FEATURES = ["content_type", "main_intent"]
print("Numeric features:", NUM_FEATURES)
print("Categorical features:", CAT_FEATURES)
print("Excluded (confirmed leaks, w03b):  avg_position_win, clicks_win, visible_queries, top_query_share, rare_share, anon_share")


Numeric features: ['word_count', 'content_age_days', 'search_volume', 'competition', 'backlinks']
Categorical features: ['content_type', 'main_intent']
Excluded (confirmed leaks, w03b):  avg_position_win, clicks_win, visible_queries, top_query_share, rare_share, anon_share


## 2. Split design

**Grouped by `client_hash_id`, `GroupShuffleSplit(test_size=0.25, random_state=42)` -- identical split to w04's baseline evaluation.** A random row-level split would let the model implicitly memorize client-specific quirks (writing style, industry, domain authority patterns) and fake skill on rows from clients it has effectively already seen. The honest question for this lane is "does this generalize to a client the model has never encountered?" -- so whole clients are held out, never split across train/test. Reusing the exact same `random_state=42` as w04 means the model and the rule baseline are evaluated on the literal same held-out pages, which is what makes the comparison table in Section 3 fair rather than a lucky-split illusion.

In [5]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(data, data["is_page_one"], data["client_hash_id"]))

train_clients = set(data.iloc[train_idx]["client_hash_id"])
test_clients  = set(data.iloc[test_idx]["client_hash_id"])
print(f"train: {len(train_idx):,} pages across {len(train_clients)} clients")
print(f"test:  {len(test_idx):,} pages across {len(test_clients)} clients")
print(f"client overlap between train and test: {len(train_clients & test_clients)}  <- must be 0")


train: 49,843 pages across 23 clients
test:  4,049 pages across 8 clients
client overlap between train and test: 0  <- must be 0


## 3. Train + compare vs my baseline

Same data, same split, same metrics (Precision@20, Precision@50, base rate) as w04 -- plus ROC AUC and Average Precision as the fuller ranking picture. One comparison table: rule baseline vs Logistic Regression vs Random Forest.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.mean(np.asarray(y_true)[order]))

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler())]), NUM_FEATURES),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CAT_FEATURES),
])

logit = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))])
rf    = Pipeline([("pre", pre), ("clf", RandomForestClassifier(n_estimators=400, class_weight="balanced",
                                                                 random_state=42, n_jobs=-1))])

FEATURE_COLS = NUM_FEATURES + CAT_FEATURES
X_train, X_test = data.iloc[train_idx][FEATURE_COLS], data.iloc[test_idx][FEATURE_COLS]
y_train, y_test = data.iloc[train_idx]["is_page_one"], data.iloc[test_idx]["is_page_one"]

logit.fit(X_train, y_train)
rf.fit(X_train, y_train)

logit_proba = logit.predict_proba(X_test)[:, 1]
rf_proba    = rf.predict_proba(X_test)[:, 1]

# Rule baseline scored on this exact test set (same rows as the model, for a fair table)
rule_test = data.iloc[test_idx]
rule_scores = rule_test["rule_score"] + (rule_test["rule_tiebreak"] / rule_test["rule_tiebreak"].max() * 0.99)


In [7]:
base_rate = y_test.mean()

rows = []
for name, scores in [
    ("Base rate (majority class)", np.full(len(y_test), y_test.mean())),
    ("Rule baseline (w04)",        rule_scores.values),
    ("Logistic Regression",        logit_proba),
    ("Random Forest",              rf_proba),
]:
    rows.append({
        "model": name,
        "Precision@20": round(precision_at_k(y_test.values, scores, 20), 3),
        "Precision@50": round(precision_at_k(y_test.values, scores, 50), 3),
        "ROC_AUC":      round(roc_auc_score(y_test, scores), 3) if name != "Base rate (majority class)" else None,
        "Avg_Precision":round(average_precision_score(y_test, scores), 3) if name != "Base rate (majority class)" else None,
    })

comparison = pd.DataFrame(rows)
print(f"held-out base rate: {base_rate:.3f}  |  test pages: {len(y_test):,}  |  test clients: {len(test_clients)}")
comparison


held-out base rate: 0.536  |  test pages: 4,049  |  test clients: 8


,model,Precision@20,Precision@50,ROC_AUC,Avg_Precision
0,Base rate (majority class),0.30,0.54,NaN,NaN
1,Rule baseline (w04),0.55,0.48,0.479,0.523
2,Logistic Regression,0.45,0.50,0.561,0.589
3,Random Forest,0.60,0.44,0.517,0.545


## 4. Errors and interpretation

What the model leans on (standardized logistic coefficients + permutation importance on the Random Forest), and a look at where it's most confidently wrong.

In [8]:
# Logistic coefficients -- signed, standardized, directly readable
feature_names = (NUM_FEATURES +
                  list(logit.named_steps["pre"].named_transformers_["cat"]
                       .named_steps["onehot"].get_feature_names_out(CAT_FEATURES)))
coefs = pd.Series(logit.named_steps["clf"].coef_[0], index=feature_names).sort_values(key=abs, ascending=False)
print("Logistic Regression coefficients (standardized -- sign = direction, magnitude = strength):")
print(coefs.head(10))


Logistic Regression coefficients (standardized -- sign = direction, magnitude = strength):
main_intent_navigational           0.577727
word_count                        -0.537892
backlinks                          0.399130
main_intent_None                  -0.358712
content_age_days                  -0.248283
content_type_comparison article    0.200530
main_intent_informational         -0.108659
content_type_keyword article      -0.067828
main_intent_transactional          0.059823
competition                       -0.039708
dtype: float64


In [9]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, scoring="average_precision",
                               n_repeats=10, random_state=42, n_jobs=-1)
perm_series = pd.Series(perm.importances_mean, index=FEATURE_COLS).sort_values(ascending=False)
print("Random Forest permutation importance (drop in Average Precision when shuffled):")
print(perm_series)


Random Forest permutation importance (drop in Average Precision when shuffled):
word_count          6.897764e-03
search_volume       2.178038e-03
competition         4.657730e-04
content_type       -1.045884e-07
backlinks          -6.006412e-04
main_intent        -8.816700e-04
content_age_days   -5.809144e-03
dtype: float64


In [10]:
# Error analysis: the model's most confidently WRONG predictions on held-out clients
errors = data.iloc[test_idx].copy()
errors["rf_proba"] = rf_proba
errors["wrong_confident"] = np.where(
    (errors["is_page_one"] == 0) & (errors["rf_proba"] > 0.8), "false_positive_confident",
    np.where((errors["is_page_one"] == 1) & (errors["rf_proba"] < 0.2), "false_negative_confident", "ok"))

wrong = errors[errors["wrong_confident"] != "ok"].sort_values("rf_proba", ascending=False)
print(f"confidently wrong predictions: {len(wrong):,} of {len(errors):,} held-out pages")
wrong[["content_hash_id", "is_page_one", "rf_proba", "wrong_confident",
       "word_count", "search_volume", "competition", "backlinks"]].head(6)


confidently wrong predictions: 644 of 4,049 held-out pages


,content_hash_id,is_page_one,rf_proba,wrong_confident,word_count,search_volume,competition,backlinks
95123,content_70b2da2cc9e29cff,0,1.0,false_positive_confident,2714,0,0.0,0
89383,content_f94fe855380e150f,0,1.0,false_positive_confident,2240,0,0.0,0
44336,content_083bd37ab9e8af05,0,1.0,false_positive_confident,3113,0,0.0,0
328,content_97a61511017b4057,0,1.0,false_positive_confident,2632,0,0.0,0
31600,content_728d1994e71b9eef,0,1.0,false_positive_confident,2461,0,0.0,0
52315,content_2b5ca79ecae24d89,0,1.0,false_positive_confident,2361,0,0.0,0


**Reading the errors and interpretation together, with real numbers:**

The point-estimate Precision@20/@50 swing a lot (Rule 0.55->0.48, RF 0.60->0.44) because the held-out test set is small -- only 8 clients, 4,049 pages -- so top-K estimates are noisy. The more stable, whole-list metrics tell a cleaner story: **Logistic Regression wins on both ROC AUC (0.561) and Average Precision (0.589)**, ahead of both the Rule baseline (0.479 / 0.523) and Random Forest (0.517 / 0.545).

**Cross-validation with the signal audit:** the logistic coefficients independently reproduce two of w04's findings -- `word_count` is strongly negative (-0.538, matching the OPPOSITE verdict: longer content associates with lower page-one odds) and `backlinks` is positive (+0.399, matching the CONFIRMED verdict). Two different methods agreeing on direction is a real robustness signal, not a coincidence. `content_age_days` is negative (-0.248) -- newer content associates with higher page-one odds, a new finding not tested in the earlier audit.

**A real limitation, found by the error analysis:** the model's most confidently wrong predictions (644 of 4,049 held-out pages) are dominated by false positives where `search_volume`, `competition`, and `backlinks` are simultaneously exactly 0 -- these are very likely pages with **missing** keyword/backlink data (the same zero-inflation w04 uncovered), not pages with genuinely zero demand. This feature set doesn't yet carry the `*_is_missing` flags built in w03b, so the model may be partly learning "no recorded data" as if it were a real, meaningful zero -- a concrete, fixable gap for a future iteration rather than a fundamental flaw.

**Framed carefully:** these are observed, associational patterns on one month, one held-out split, of one company's pseudonymized portfolio -- not causal claims, and given the small held-out client count, not yet validated for stability across splits either. That stability check is exactly the job of the next notebook (validation audit).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.